# 03a. 네 가지 흐름 ①: 순차와 병렬

앞 노트북에서 Supervisor와 Handoff를 실행해 비교했습니다. 이름이 다른 구조를 여럿 보았지만, 실행 단계에서 그래프가 할 수 있는 일은 많지 않습니다. 앞 작업이 끝나야 다음으로 가거나(순차), 동시에 돌리거나(병렬), 조건에 따라 고르거나(분기), 되돌아갑니다(루프). 이 과정은 패턴 이름을 잠시 치워 두고 네 흐름을 직접 만든 다음 패턴 이름을 네 흐름으로 번역합니다.

이 노트북 **03a**는 그중 **업무의 선행 조건만으로 정해지는 두 흐름**, 순차와 병렬을 다룹니다. 값을 보고 다음을 정하는 분기와 루프, 그리고 패턴 이름의 번역은 **03b**에서 합니다.

## 학습 목표

- 패턴 이름 없이 업무의 상태, 작업, 선행 조건, 종료 조건을 먼저 표로 씁니다.
- 순차와 병렬을 `StateGraph`로 직접 만들고 각각의 실패를 재현합니다.
- `Send`와 reducer의 역할을 설명하고, 같은 분해에서만 병렬의 시간 효과를 비교합니다.
- 최장 경로로 지연을 계산하고 실측과 대조합니다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 1 | 패턴 말고 업무 먼저 | 7 |
| 2 | 순차와 Context | 7 |
| 3 | 병렬, reducer, 최장 경로 | 12 |
| 4 | 직접 해보기와 정리 | 4 |

## 호출 규모

이 노트북은 **실제 모델**만 씁니다. `Day03_new/.env`의 `ANTHROPIC_API_KEY`가 없으면 첫 코드 셀에서 안내와 함께 멈춥니다. 유료 셀은 실행 직전에 규모를 한 줄로 알려 줍니다. 비용은 본인 키로 청구됩니다.

그래프의 구조, reducer, 시간 측정처럼 모델의 판단이 필요 없는 부분은 일반 함수 노드로 돌립니다. 이런 셀은 대부분 출력에 `모델 호출 0회`를 적습니다. 준비 셀, 그래프를 만들거나 그리기만 하는 셀, 앞 실행의 기록을 표로 읽기만 하는 셀은 이 문구가 없어도 모델을 부르지 않습니다.

| 구분 | 유료 셀 수 | 모델 호출(추정) |
|---|---:|---:|
| 조사 → 작성 → 검토 | 1 | 5 |
| 실제 Worker 3개의 병렬과 직렬 | 2 | 12~24 |
| 기본 실행 합계 | 3 | 약 17~29 |
| 해보기 3을 켜면 더해지는 호출 | 1 | 4~8 |

모델 호출 수는 실행마다 달라집니다. 모델이 도구를 몇 번 쓰는지는 모델이 정하기 때문입니다.

In [ ]:
import json
import operator
import time
from typing import Annotated, Literal, TypedDict

from deepagents import create_deep_agent
from langchain.agents.structured_output import ProviderStrategy
from langchain_core.runnables import RunnableConfig
from langgraph.errors import GraphRecursionError, InvalidUpdateError
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, Send
from pydantic import BaseModel

from mas_lab.contracts import FinalAnswer, Route, WorkerResult, WorkerTask, join_results, merge_results, plan_hash, validate_final
from mas_lab.desk import Desk, load_billing, load_outage
from mas_lab.hybrid import ROLE_GOALS, ROLES
from mas_lab.runner import make_desk
from mas_lab.structures import OUTCOME_GUIDE, gate_answer, scope_line
from mas_lab.trace import ModelMeter, Trace, longest_path, overlap_seconds, timeline
from mas_lab.view import flows_of, show_graph, show_table, write_design_log
from mas_lab.workers import DENY_WRITES, ROLE_TOOLS, allowlist_middleware, contract_prompt, run_worker
from rag_lab.corpus import load_cases
from rag_lab.evidence import BudgetExceeded
from rag_lab.settings import announce, key_status, make_model, require_key

settings = require_key()
model = make_model()
CASES = {case["id"]: case for case in load_cases()}
T1001 = CASES["complex-double-pay"]
desk: Desk = make_desk(T1001, label="점검")
print(key_status(settings))
print("고객 요청 T-1001:", T1001["question"])
print("고객:", desk.scope.entity, "/ 기준일:", desk.scope.as_of, "/ 결제 건수:", len(desk.payments()))
print("측정 장치:", Trace.__name__, ModelMeter.__name__)

## 1절. 패턴 말고 업무 먼저

새 업무를 받으면 "Billing Agent, Policy Agent, Supervisor를 두자"부터 떠올리기 쉽습니다. 그런데 이 문장에는 업무에서 무엇이 먼저 필요한지가 없습니다. 패턴 이름을 먼저 고르면 그 뒤로는 업무를 패턴에 맞추게 됩니다. 그래서 다음 순서로 먼저 씁니다.

1. **목표.** 어떤 상태가 되면 끝인가요?
2. **필요한 상태.** 그 목표에 어떤 값들이 있어야 하나요?
3. **작업.** 그 값은 어떤 일을 해야 생기나요?
4. **선행 조건.** 그 일은 무엇이 끝나야 시작할 수 있나요?
5. **종료 조건.** 그 일은 언제 끝났다고 판단하나요?

순서가 이렇게 정해지면 구조는 표에서 읽힙니다. 선행 조건이 있는 작업은 순차이고, 선행 조건이 없는 작업끼리는 동시에 실행할 수 있습니다.

**먼저 예측하세요.** 고객 요청 T-1001은 "결제가 두 번 됐어요. 취소하고 보상안도 함께 확인해 주세요."입니다. 아래 셀을 실행하기 전에 작업을 종이에 쓰고, 작업마다 선행 작업을 적어 보세요. 크레딧 금액은 어떤 값 세 개가 있어야 계산할 수 있나요? 결제 사실과 환불 규정 중 무엇이 먼저 끝나야 하나요, 아니면 순서가 상관없나요?

In [ ]:
TASKS = {
    "verify":   {"needs": [], "state": "customer_verified", "done": "고객이 범위 안인지 정해졌다"},
    "billing":  {"needs": ["verify"], "state": "billing_facts", "done": "같은 청구서의 결제가 모두 조회됐다"},
    "policy":   {"needs": ["verify"], "state": "policy_evidence", "done": "환불 규정, SLA, 추가 약정의 원문을 찾았다"},
    "outage":   {"needs": ["verify"], "state": "outage_facts", "done": "해당 월 가용률이 조회됐다"},
    "credit":   {"needs": ["billing", "policy", "outage"], "state": "credit_amount", "done": "기본 이용료 x 비율을 계산했다"},
    "proposal": {"needs": ["credit"], "state": "proposal", "done": "제안과 근거, 불확실성을 썼다"},
    "gate":     {"needs": ["proposal"], "state": "gate_decision", "done": "코드 게이트가 허용, 승인, 거절을 정했다"},
}

def plan_batches(tasks: dict) -> list[list[str]]:
    """위상 정렬. 각 묶음은 앞 묶음이 끝나면 동시에 실행할 수 있는 작업입니다."""
    missing = sorted({need for task in tasks.values() for need in task["needs"]} - set(tasks))
    if missing:
        raise ValueError(f"없는 작업에 의존합니다: {missing}")
    done, batches = set(), []
    while len(done) < len(tasks):
        ready = sorted(name for name, task in tasks.items() if name not in done and set(task["needs"]) <= done)
        if not ready:
            raise ValueError(f"순환 의존입니다: {sorted(set(tasks) - done)}")
        batches.append(ready)
        done |= set(ready)
    return batches

show_table([{"작업": name, "선행 작업": ", ".join(t["needs"]) or "없음", "채우는 상태 칸": t["state"], "종료 조건": t["done"]} for name, t in TASKS.items()])
for number, batch in enumerate(plan_batches(TASKS), 1):
    print(f"묶음 {number}: {batch}")
print("모델 호출 0회")

In [ ]:
# 실패 재현: 형식은 맞지만 실행할 수 없는 계획을 넣습니다
broken_plans = {
    "순환 의존": {**TASKS, "verify": {**TASKS["verify"], "needs": ["gate"]}},
    "없는 작업 의존": {**TASKS, "credit": {**TASKS["credit"], "needs": ["billing", "tax_lookup"]}},
}
for label, plan in broken_plans.items():
    print(f"{label}: json으로 직렬화됨 = {bool(json.dumps(plan, ensure_ascii=False))}")
    try:
        print("  묶음:", plan_batches(plan))
    except ValueError as error:
        print(f"  거절: {type(error).__name__}: {error}")
print("모델 호출 0회")

**관찰할 것.** 표 전체를 다시 읽으며 네 흐름이 어디 있는지 표시해 보세요.

- 순차: `선행 작업` 칸이 차 있는 행입니다. `credit`의 선행 작업이 세 개라는 점을 보세요.
- 병렬: `묶음` 출력에서 한 줄에 둘 이상 놓인 작업입니다. 이 묶음이 곧 "동시에 돌려도 되는 후보"입니다. 돌려야 한다는 뜻은 아닙니다.
- 분기: 표에 칸은 없지만 `verify`의 종료 조건("범위 안인지 정해졌다")과 `gate`의 종료 조건("허용, 승인, 거절")은 값에 따라 다음 작업이 달라집니다.
- 루프: 이 표에는 아직 없습니다. `종료 조건`이 만족되지 않은 작업을 다시 시도하려면 어떤 칸이 더 필요할지 생각해 보세요.

실패 재현 출력에서는 두 계획 모두 `json으로 직렬화됨`이 `True`인데 거절되었습니다. 모델이 계획을 JSON으로 냈다는 것과 그 계획이 실행 가능하다는 것은 다른 사실입니다. 형식 검사는 스키마가, 실행 가능성 검사는 이 위상 정렬 같은 코드가 맡습니다.

다음 절부터 이 표의 일부를 그래프로 옮깁니다.

### 그림으로 보기: T-1001의 선행 조건

위 `TASKS` 표와 `묶음` 출력을 그림으로 옮기면 다음과 같습니다. 화살표는 `needs`, 곧 선행 조건입니다. 묶음은 `plan_batches`가 선행 조건만으로 계산한 결과입니다.

![T-1001 업무의 선행 조건. verify가 끝나면 billing, policy, outage를 동시에 시작할 수 있고, 셋이 모두 끝나야 credit, proposal, gate가 차례로 이어진다](./assets/03a-1-task-graph.png)

- 묶음 2의 세 작업 사이에는 화살표가 없습니다. 서로 결과가 필요 없다는 뜻이고, 그래서 "동시에 돌려도 되는 후보"입니다.
- `credit`으로 들어오는 화살표는 셋입니다. 셋 중 하나라도 끝나지 않으면 `credit`은 시작할 수 없습니다. 이 모양이 순차입니다.

## 2절. 순차와 Context

**문제.** 선행 결과 없이 다음 작업이 돌면 안 됩니다. 그리고 순차로 잇기만 해서는 충분하지 않습니다. 다음 작업이 무엇을 받아야 하는지를 정해야 합니다.

`StateGraph`는 세 가지로 이루어집니다.

| 요소 | 역할 | T-1001에서 |
|---|---|---|
| State | 작업들이 함께 읽고 쓰는 칸 | `billing_facts`, `credit` |
| Node | State를 받아 바뀐 칸만 돌려주는 함수 | `verify`, `billing`, `credit` |
| Edge | 다음에 어느 Node를 실행할지 | `billing → credit` |

Node가 전체 State가 아니라 **바뀐 칸만** 돌려준다는 점이 중요합니다. `stream(stream_mode="updates")`로 돌리면 Node별로 돌려준 칸만 한 줄씩 나옵니다. 그래서 실행 도중 상태가 어떻게 쌓이는지를 그대로 읽을 수 있습니다.

아래 그래프의 크레딧 비율은 `data/business/sla-a-v2.md`의 "99.0% 이상 99.5% 미만" 구간 값 15를 상수로 넣었습니다. 이 절은 흐름을 보는 자리여서 비율을 코드에 고정했습니다. 실제 판단에서는 Worker가 원문을 읽고 비율을 고르고, 이 뒤의 절에서 그렇게 돌립니다.

In [ ]:
class SeqState(TypedDict, total=False):
    entity: str
    customer_verified: bool
    billing_facts: dict
    credit: int

CREDIT_PERCENT = 15  # data/business/sla-a-v2.md의 99.0% 이상 99.5% 미만 구간

def verify(state):
    return {"customer_verified": bool(load_billing(state["entity"]))}

def billing(state):
    row = next((p for p in load_billing(state["entity"]) if p["invoice_id"] == "INV-2026-08"), None)
    return {"billing_facts": {"base_fee": row["base_fee_part"] if row else 0}}

def credit(state):
    return {"credit": state["billing_facts"]["base_fee"] * CREDIT_PERCENT // 100}

builder = StateGraph(SeqState)
for name, node in [("verify", verify), ("billing", billing), ("credit", credit)]:
    builder.add_node(name, node)
builder.add_edge(START, "verify")
builder.add_edge("verify", "billing")
builder.add_edge("billing", "credit")
builder.add_edge("credit", END)
sequence_app = builder.compile()

for update in sequence_app.stream({"entity": "알파"}, stream_mode="updates"):
    print(update)
print("모델 호출 0회")

### 그림으로 보기: State에 칸이 쌓이는 모습

방금 출력된 `updates` 세 줄을 State의 칸으로 옮기면 다음과 같습니다. `…` 자리의 값은 실행할 때 코드가 계산한 값입니다.

![순차 그래프의 State 변화. 단계마다 Node가 돌려준 칸 하나만 새로 채워지고 앞 단계의 칸은 그대로 이어진다](./assets/03a-2-state-updates.png)

- 각 Node는 State 전체가 아니라 **자기가 바꾼 칸 하나만** 돌려줍니다. 그래서 `updates` 출력 한 줄이 그 단계가 State에 한 일의 전부입니다.
- `credit` Node는 `billing_facts`를 읽습니다. 앞 단계가 State에 쓴 칸이 다음 Node의 입력이 되는 것, 이것이 순차에서 "전달"입니다.

In [ ]:
# 실패 재현: 범위 밖 고객에게도 고정 간선은 다음 단계를 돌립니다
def walk(app, initial: dict, **config) -> tuple[list[str], dict]:
    """실행 경로와 마지막 상태를 돌려줍니다. stream의 키가 곧 실행된 Node입니다."""
    path, state = [], dict(initial)
    for update in app.stream(initial, stream_mode="updates", config=config or None):
        for node, change in update.items():
            path.append(node)
            state.update(change or {})
    return path, state

builder = StateGraph(SeqState)
for name, node in [("verify", verify), ("billing", billing), ("credit", credit)]:
    builder.add_node(name, node)
builder.add_edge(START, "verify")
builder.add_conditional_edges("verify", lambda s: "billing" if s["customer_verified"] else END, ["billing", END])
builder.add_edge("billing", "credit")
builder.add_edge("credit", END)
guarded_app = builder.compile()

rows = []
for label, app in [("고정 간선", sequence_app), ("조건부 간선", guarded_app)]:
    for entity in ["알파", "감마"]:
        path, state = walk(app, {"entity": entity})
        rows.append({"그래프": label, "고객": entity, "경로": " → ".join(path), "credit": state.get("credit", "계산 안 함")})
show_table(rows)
print("모델 호출 0회")

순차에서 Context를 나누는 이유를 봅시다. 조사, 작성, 검토를 한 역할이 모두 하면 원문 전체가 답을 쓰는 호출까지 따라갑니다. 역할을 나눈다는 말은 모델을 여러 개 쓴다는 뜻이 아닙니다. **입력을 나눈다**는 뜻입니다.

| 역할 | 받는 것 | 받지 않는 것 | 돌려주는 것 |
|---|---|---|---|
| 조사자 | 고객 요청, 결제와 가용률 사실, 원문 | 없음 | 출처가 붙은 조사 메모 |
| 작성자 | 고객 요청, 조사 메모 | 원문, 조사자의 내부 과정 | 고객에게 보낼 초안 |
| 검토자 | 조사 메모, 초안 | 원문 | 통과 또는 반려, 그 이유 |

표의 `조사 메모`가 역할 사이의 전달물입니다. 이 전달물이 부실하면 뒤의 역할은 알 방법이 없습니다. 그래서 같은 그래프를 메모를 넘기는 경우와 넘기지 않는 경우로 돌립니다. 조건을 하나만 바꾸기 위해 두 번째 실행은 첫 실행의 메모를 재사용하고, 작성자에게 메모를 주지 않습니다.

In [ ]:
class RoleState(TypedDict, total=False):
    question: str
    sources: str
    notes: str
    pass_notes: bool
    draft: str
    review: str

ROLE_RECORDS = []

def call_role(role: str, prompt: str, run: str) -> str:
    meter, started = ModelMeter(), time.perf_counter()
    reply = model.invoke(prompt, config={"callbacks": [meter]})
    totals = meter.totals()
    ROLE_RECORDS.append({"실행": run, "역할": role, "입력 바이트": len(prompt.encode()), "입력 토큰": totals["input_tokens"],
                         "출력 토큰": totals["output_tokens"], "시간(초)": round(time.perf_counter() - started, 2)})
    return reply.text

def researcher(state):
    if state.get("notes"):
        return {}
    return {"notes": call_role("조사자", f"조사자입니다. 자료에서 고객 요청에 답하는 데 필요한 사실만 항목으로 뽑고 각 항목 끝에 evidence_id나 결제 번호를 붙이세요. 답변은 쓰지 마세요.\n고객 요청: {state['question']}\n자료: {state['sources']}", "메모 전달")}

def writer(state):
    run = "메모 전달" if state["pass_notes"] else "메모 없음"
    notes = state["notes"] if state["pass_notes"] else "전달받은 메모가 없습니다."
    return {"draft": call_role("작성자", f"작성자입니다. 조사 메모의 사실만 써서 고객에게 보낼 답을 다섯 줄 이내로 쓰세요. 메모에 없으면 '확인 필요'로 쓰세요.\n고객 요청: {state['question']}\n조사 메모: {notes}", run)}

def reviewer(state):
    run = "메모 전달" if state["pass_notes"] else "메모 없음"
    return {"review": call_role("검토자", f"검토자입니다. 첫 줄에 '통과' 또는 '반려'만 쓰고, 다음 줄부터 조사 메모의 사실 중 초안에서 빠진 것과 메모에 없는 문장을 항목으로 쓰세요.\n조사 메모: {state['notes']}\n초안: {state['draft']}", run)}

builder = StateGraph(RoleState)
for name, node in [("researcher", researcher), ("writer", writer), ("reviewer", reviewer)]:
    builder.add_node(name, node)
builder.add_edge(START, "researcher")
builder.add_edge("researcher", "writer")
builder.add_edge("writer", "reviewer")
builder.add_edge("reviewer", END)
role_app = builder.compile()
print("역할 그래프:", [e for e in role_app.get_graph().nodes if not e.startswith("__")])

In [ ]:
announce("조사 → 작성 → 검토, 작성자 메모 유무 비교", model_calls=5)
role_desk = make_desk(T1001, label="역할")
hits = [e for query in ["중복 결제 환불 규정", "월간 가용률 서비스 크레딧 비율", "서비스 크레딧 신청 기한"] for e in role_desk.context.search(query, limit=2)["evidence"]]
sources = json.dumps({"결제": [p for p in role_desk.payments() if p["invoice_id"] == "INV-2026-08"], "가용률": load_outage("알파")["2026-08"],
                      "원문": [{"evidence_id": e["evidence_id"], "quote": e["quote"]} for e in hits]}, ensure_ascii=False)

with_notes = role_app.invoke({"question": T1001["question"], "sources": sources, "pass_notes": True})
without_notes = role_app.invoke({"question": T1001["question"], "sources": sources, "notes": with_notes["notes"], "pass_notes": False})

for label, result in [("메모 전달", with_notes), ("메모 없음", without_notes)]:
    print(f"--- {label} ---\n[초안]\n{result['draft']}\n[검토]\n{result['review']}\n")

In [ ]:
show_table(ROLE_RECORDS)
print("자료 바이트:", len(sources.encode()), "/ 조사 메모 바이트:", len(with_notes["notes"].encode()))

**관찰할 것.** 역할의 수가 아니라 **전달물**을 읽습니다.

- 표 아래의 `자료 바이트`와 `조사 메모 바이트`를 비교하세요. 메모가 자료를 얼마나 줄였나요? 줄지 않았거나 오히려 늘었다면 이번 분업이 얻은 것은 입력 절약이 아니라 책임 분리뿐입니다. 메모를 더 짧게 받으려면 조사자의 지침에 무엇을 더해야 할까요?
- `메모 없음`의 초안을 읽으세요. 작성자가 '확인 필요'로 썼다면 지침대로 멈춘 것이고, 금액이나 날짜를 썼다면 근거 없이 지어낸 것입니다. 어느 쪽이 나왔는지 적으세요. 어느 쪽이든 이 작성자는 자기에게 무엇이 빠졌는지 알 방법이 없습니다.
- 검토자의 첫 줄을 두 실행에서 비교하세요. `메모 전달`에서도 반려가 나왔다면 빠진 항목을 읽으세요. 고객 답에 없어도 되는 세부 사항까지 빠졌다고 적었다면, 초안이 나빠서가 아니라 검토 기준이 "메모의 사실을 모두 담았는가"이기 때문입니다. 다섯 줄 답에 이 기준을 걸면 대부분 반려됩니다. 역할 사이의 기준도 전달물의 일부입니다. 검토자가 원문을 받지 않으므로 메모 자체가 틀린 경우는 잡지 못한다는 점도 기억하세요.
- 호출은 세 번이고 각 호출의 입력 토큰이 모두 비용입니다. 역할을 나누면 호출마다 입력은 작아지지만 호출 수는 늘어납니다. 합계를 직접 더해 보세요.

여기까지는 한 번에 한 작업만 돌았습니다. 1절의 `묶음 2`에는 작업이 세 개 있었습니다. 이것을 동시에 돌리면 무엇이 달라지는지가 다음 절의 질문입니다.

## 3절. 병렬, reducer, 최장 경로

**문제.** `billing`, `policy`, `outage`는 서로의 결과가 필요 없습니다. 그러면 동시에 돌려서 기다리는 시간을 줄일 수 있을까요? 동시에 돌리려면 두 가지가 정해져야 합니다. 여러 작업이 같은 칸에 쓰면 어떻게 합칠까요? 그리고 정말 동시에 돌았다는 것을 어떻게 확인할까요?

**먼저 예측하세요.** 실행 전에 두 가지를 적어 보세요.

1. 세 작업이 reducer 없는 일반 칸 `last`에 동시에 쓰면 어떻게 될까요? 마지막에 쓴 값이 남을까요, 오류가 날까요?
2. 0.3초짜리 작업 세 개를 동시에 돌리면 전체 시간은 몇 초일까요? `for` 문으로 돌리면요?

### `Send`를 처음부터

지금까지의 간선은 그래프를 만들 때 정해졌습니다. 몇 개의 작업을 돌릴지 실행해 봐야 안다면 간선을 미리 그릴 수 없습니다. `Send`가 이 문제를 풉니다.

`Send("노드 이름", 입력)`은 "이 Node를, 이 입력으로, 한 번 더 실행하라"는 **작업 표 한 장**입니다. 쓰는 방법은 세 단계입니다.

1. 조건부 간선의 함수가 `Send`의 **목록**을 돌려줍니다. 목록의 길이가 곧 작업 수입니다.
2. `add_conditional_edges`의 셋째 인자에 도착할 수 있는 Node 이름을 적습니다. `get_graph`가 이 목록으로 간선을 그립니다.
3. 받는 Node의 첫 인자는 전체 State가 아니라 `Send`에 넣은 **입력**입니다. 그래서 Worker마다 필요한 것만 줄 수 있습니다.

같은 때 만들어진 `Send`는 그래프의 같은 단계(superstep)에서 함께 실행되고, 그 단계가 모두 끝나야 다음 Node로 넘어갑니다. 각 실행은 돌려준 칸을 State에 씁니다. 여기서 앞의 첫 번째 예측이 나옵니다. 한 단계에서 같은 칸에 여러 값이 오면 어떻게 될까요?

In [ ]:
# 실패 재현: reducer 없는 칸에 Send 세 개가 동시에 씁니다
class PlainState(TypedDict, total=False):
    plan: dict
    last: str

def fan_out_plain(state):
    return [Send("write_last", {"task_id": name}) for name in state["plan"]]

def write_last(payload: dict):
    return {"last": payload["task_id"]}

builder = StateGraph(PlainState)
builder.add_node("write_last", write_last)
builder.add_conditional_edges(START, fan_out_plain, ["write_last"])
builder.add_edge("write_last", END)
plain_app = builder.compile()

try:
    plain_app.invoke({"plan": {"billing": 0.3, "policy": 0.3, "outage": 0.3}})
except InvalidUpdateError as error:
    print(f"{type(error).__name__}: {str(error).splitlines()[0]}")
print("모델 호출 0회")

In [ ]:
WORK = {"billing": 0.3, "policy": 0.3, "outage": 0.3}

trace, started = Trace("for 문"), time.perf_counter()
for task_id, seconds in WORK.items():
    with trace.span(task_id):
        time.sleep(seconds)
print(f"for 문: 전체 {time.perf_counter() - started:.3f}초, 겹침 {overlap_seconds(trace.spans):.3f}초")
print(timeline(trace.spans))
print("모델 호출 0회")

오류 문장은 `At key 'last'`로 시작합니다. 어느 칸이 문제인지 알려 주고, 한 단계에 값을 하나만 받을 수 있다고 말합니다. 해결은 그 칸에 **합치는 규칙**을 붙이는 것입니다. 이 규칙이 reducer이고 `Annotated[타입, 합치는 함수]`로 적습니다. 여기서는 `merge_results`를 씁니다. 작업 ID를 키로 삼아 dict를 합칩니다. 왜 단순히 리스트에 이어 붙이지 않는지는 이 절의 뒷부분에서 직접 호출해 봅니다.

`for` 문 출력은 비교 기준입니다. 막대가 계단처럼 이어지면 겹침이 없다는 뜻입니다. 빠르게 도는 `for` 문을 병렬로 착각하지 마세요. **세로로 겹치는 칸이 있어야** 동시 실행입니다.

아래 Node는 `config`를 둘째 인자로 받습니다. 실행 때 넘기는 값을 읽는 통로입니다. `Trace`를 여기로 넘기면 그래프를 다시 만들지 않고도 실행마다 새 기록을 쓸 수 있습니다. 이것이 잠시 뒤의 비교에서 중요해집니다.

### 그림으로 보기: `Send`와 reducer

`Send`로 병렬을 만드는 흐름을 그림으로 옮기면 다음과 같습니다. 아래 코드 셀의 `dispatch`, `work`, `FanState`가 이 그림의 세 칸입니다.

![Send로 같은 Node를 세 번 실행하는 흐름. dispatch가 Send 세 개를 돌려주면 work가 입력만 달리해 세 번 실행되고, 한 단계가 끝나면 reducer가 붙은 results 칸에 합쳐진다](./assets/03a-3-send-fanout.png)

- `dispatch`가 돌려주는 것은 실행 결과가 아니라 **작업 표**(`Send`) 세 장입니다. 목록의 길이가 곧 작업 수입니다.
- 세 `work`는 같은 Node를 입력만 바꿔 실행한 것이고 같은 단계에서 함께 돕니다.
- 세 실행이 모두 같은 `results` 칸에 씁니다. 이 칸에 합치는 규칙(`merge_results`)이 붙은 이유가 앞의 `InvalidUpdateError`입니다.

In [ ]:
class FanState(TypedDict, total=False):
    plan: dict
    results: Annotated[dict, merge_results]

def dispatch(state):
    return [Send("work", {"task_id": name, "seconds": seconds}) for name, seconds in state["plan"].items()]

def work(payload: dict, config: RunnableConfig):
    with config["configurable"]["trace"].span(payload["task_id"]):
        time.sleep(payload["seconds"])
    return {"results": {payload["task_id"]: {"seconds": payload["seconds"]}}}

builder = StateGraph(FanState)
builder.add_node("work", work)
builder.add_conditional_edges(START, dispatch, ["work"])
builder.add_edge("work", END)
fan_app = builder.compile()

def run_fan(plan: dict, **options) -> dict:
    trace, started = Trace("fan"), time.perf_counter()
    state = fan_app.invoke({"plan": plan}, config={"configurable": {"trace": trace}, **options})
    return {"seconds": round(time.perf_counter() - started, 3), "spans": trace.spans, "results": state["results"], "graph_id": id(fan_app), "options": options}

print("Send 입력(최소 입력입니다. plan 전체가 Worker에게 가지 않습니다):")
for item in dispatch({"plan": WORK}):
    print(" ", item.node, item.arg)
show_graph(fan_app)

In [ ]:
fan_parallel = run_fan(WORK)
print(f"Send: 전체 {fan_parallel['seconds']:.3f}초, 겹침 {overlap_seconds(fan_parallel['spans']):.3f}초")
print(timeline(fan_parallel["spans"]))
print("합쳐진 결과:", fan_parallel["results"])
print("모델 호출 0회")

In [ ]:
fan_serial = run_fan(WORK, max_concurrency=1)
show_table([
    {"실행": label, "config 차이": run["options"] or "없음", "plan_hash": plan_hash(WORK), "전체(초)": run["seconds"], "겹침(초)": overlap_seconds(run["spans"])}
    for label, run in [("병렬", fan_parallel), ("직렬", fan_serial)]
])
print(timeline(fan_serial["spans"]))
print("같은 그래프 객체:", fan_parallel["graph_id"] == fan_serial["graph_id"], "/ 그래프를 고치지 않았습니다. 바뀐 것은 config의 max_concurrency 하나입니다.")
print("모델 호출 0회")

`max_concurrency`는 한 단계에서 동시에 도는 작업 수의 상한입니다. 값 1이면 `Send` 세 개가 차례로 돕니다. 이 스위치가 있어서 **분해를 똑같이 두고 동시성만 바꾼 비교**를 할 수 있습니다. `plan_hash`는 계획을 정렬된 JSON으로 만들어 해시한 값이고, 두 실행의 분해가 같다는 증거로 씁니다. 병렬과 직렬을 다른 코드로 만들어 비교하면 작업 수나 입력이 슬쩍 달라져도 알아채기 어렵습니다.

이번에는 작업 시간을 고르지 않게 바꿉니다. 병렬로 돌렸을 때 걸리는 시간은 작업 시간의 합이 아니라 **최장 경로**, 곧 의존을 지키면서 가장 오래 걸리는 줄의 길이입니다. `longest_path(시간, 의존)`이 이것을 계산합니다. 실측과 나란히 놓고 확인합니다.

In [ ]:
UNEVEN = {"billing": 0.2, "policy": 0.2, "outage": 1.0}
uneven_parallel, uneven_serial = run_fan(UNEVEN), run_fan(UNEVEN, max_concurrency=1)
calculated = longest_path(UNEVEN, {})
show_table([
    {"실행": "병렬", "계산(초)": calculated, "계산식": "longest_path", "실측(초)": uneven_parallel["seconds"], "겹침(초)": overlap_seconds(uneven_parallel["spans"])},
    {"실행": "직렬", "계산(초)": round(sum(UNEVEN.values()), 3), "계산식": "합계", "실측(초)": uneven_serial["seconds"], "겹침(초)": overlap_seconds(uneven_serial["spans"])},
])
print(timeline(uneven_parallel["spans"]))
print("credit까지 잇는 경우의 계산:", longest_path({**UNEVEN, "credit": 0.1}, {"credit": list(UNEVEN)}))
print("모델 호출 0회")

### 그림으로 보기: 합계와 최장 경로

위 표의 `계산(초)` 칸이 이 그림과 같은 계산입니다. 그림의 숫자는 측정값이 아니라 `UNEVEN`의 값과 `longest_path`, 합계를 계산한 것입니다. 측정값은 위 표의 `실측(초)`에서 읽으세요.

![작업 시간이 0.2, 0.2, 1.0초일 때 차례로 돌리면 합계 1.4초이고 동시에 돌리면 최장 경로 1.0초이며, credit이 셋을 기다리면 1.1초다](./assets/03a-4-longest-path.png)

- 차례로 돌리면 시간은 합계입니다.
- 동시에 돌리면 가장 긴 작업 하나(`outage`)가 전체 시간을 정합니다. 짧은 두 작업을 동시에 돌려서 줄어드는 시간은 그 두 작업이 차지하던 시간뿐입니다.
- `credit`이 세 작업을 모두 기다리면 가장 긴 작업 뒤에 이어 붙습니다. 의존이 있는 작업은 동시에 돌릴 수 없으므로 시간이 더해집니다.

In [ ]:
# 실패 재현: 의존을 무시하고 credit을 함께 Send합니다
class FactState(TypedDict, total=False):
    results: Annotated[dict, merge_results]
    credit: int

def fact_work(payload: dict):
    known = {key: value for fact in payload["known"].values() for key, value in fact.items()}
    facts = {"billing": lambda: {"base_fee": load_billing("알파")[0]["base_fee_part"]}, "policy": lambda: {"percent": CREDIT_PERCENT},
             "outage": lambda: {"availability": load_outage("알파")["2026-08"]["availability_percent"]},
             "credit": lambda: {"credit": known.get("base_fee", 0) * known.get("percent", 0) // 100, "입력": known}}
    return {"results": {payload["task_id"]: facts[payload["task_id"]]()}}

def build_facts(fan: list[str], after: bool):
    def send_all(state):
        return [Send("fact_work", {"task_id": name, "known": state.get("results", {})}) for name in fan]
    def credit_after(state):
        return fact_work({"task_id": "credit", "known": state["results"]})
    builder = StateGraph(FactState)
    builder.add_node("fact_work", fact_work)
    builder.add_conditional_edges(START, send_all, ["fact_work"])
    builder.add_node("credit_after", credit_after)
    builder.add_edge("fact_work", "credit_after" if after else END)
    builder.add_edge("credit_after", END)
    return builder.compile()

together = build_facts(["billing", "policy", "outage", "credit"], after=False).invoke({})
ordered = build_facts(["billing", "policy", "outage"], after=True).invoke({})
show_table([{"그래프": "함께 Send", "credit": together["results"]["credit"]["credit"], "credit이 받은 입력": together["results"]["credit"]["입력"]},
            {"그래프": "Send 뒤에 순차", "credit": ordered["results"]["credit"]["credit"], "credit이 받은 입력": ordered["results"]["credit"]["입력"]}])
print("모델 호출 0회")

In [ ]:
# 실패 재현: 같은 작업 ID에 다른 값이 오면 merge_results가 거절합니다
first = {"outage": {"availability": 99.3}}
print("재전달(같은 값):", merge_results(first, {"outage": {"availability": 99.3}}))
print("다른 작업 추가:", merge_results(first, {"billing": {"base_fee": 1000000}}))
try:
    merge_results(first, {"outage": {"availability": 99.95}})
except ValueError as error:
    print(f"상충(다른 값): {type(error).__name__}: {error}")
print("모델 호출 0회")

**관찰할 것.** 여기까지는 모델 호출이 없어서 값이 거의 고정되어 있습니다.

- `credit`이 받은 입력 칸을 읽으세요. 함께 `Send`한 경우 입력이 빈 dict이고, 오류 없이 `credit` 값이 나왔습니다. **`Send`는 의존을 모릅니다.** 작업 표를 나눠 줄 뿐입니다. 의존은 그래프의 단계로 직접 나눠야 합니다. 이 실패는 예외가 나지 않아서 금액을 보는 사람이 없으면 그대로 지나갑니다.
- `merge_results`는 같은 값의 재전달을 통과시키고, 상충은 `ValueError`로 멈춥니다. 재시도나 재전달은 분산 실행에서 흔합니다. 리스트에 이어 붙이는 reducer였다면 같은 결과가 두 번 쌓이고 상충도 조용히 함께 남습니다.
- 최장 경로 표에서 병렬의 `실측`이 `계산`보다 조금 긴 만큼이 그래프 자체의 부담입니다. 가장 긴 작업 하나가 전체를 정한다면, 그 작업을 줄이지 않고 짧은 작업을 병렬로 돌리는 일의 이득은 얼마일까요?

한 가지 가설을 세우고 다음 실측으로 확인합니다. **병렬이 줄이는 것은 전체 시간뿐이고, 합산 호출과 토큰은 줄지 않습니다.** 잠만 자는 함수로는 이 가설을 시험할 수 없으니 실제 Worker를 씁니다.

아래의 `PLAN`은 코드에 고정한 분해입니다. Worker 세 개를 같은 입력으로, 같은 그래프 객체에서 돌립니다. 달라지는 것은 `max_concurrency`뿐이며, 두 실행은 각자 새 `Desk`를 받아 예산과 근거가 섞이지 않습니다.

In [ ]:
PLAN = {"case_id": T1001["id"], "question": T1001["question"],
        "tasks": [{"task_id": role, "goal": f"{ROLE_GOALS[role]}\n고객 요청: {T1001['question']}", "allowed_tools": sorted(ROLE_TOOLS[role])} for role in ROLES]}

class PlanState(TypedDict, total=False):
    tasks: list[dict]
    results: Annotated[dict, merge_results]

def real_worker(payload: dict, config: RunnableConfig):
    task = WorkerTask.model_validate(payload["task"])
    try:
        result = run_worker(task, config["configurable"]["desk"], model)
    except BudgetExceeded as error:  # 예산 초과도 결과 행으로 남깁니다.
        result = WorkerResult(task_id=task.task_id, status="failed", error=str(error))
    return {"results": {task.task_id: result.model_dump()}}

def build_plan_app(worker):
    builder = StateGraph(PlanState)
    builder.add_node("worker", worker)
    builder.add_conditional_edges(START, lambda state: [Send("worker", {"task": t}) for t in state["tasks"]], ["worker"])
    builder.add_edge("worker", END)
    return builder.compile()

plan_app = build_plan_app(real_worker)

def run_plan(plan: dict, label: str, app=plan_app, **options) -> dict:
    desk, meter, started = make_desk(T1001, label=label), ModelMeter(), time.perf_counter()
    state = app.invoke({"tasks": plan["tasks"]}, config={"callbacks": [meter], "configurable": {"desk": desk}, **options})
    return {"label": label, "plan_hash": plan_hash(plan), "results": state["results"], "desk": desk, "meter": meter, "seconds": round(time.perf_counter() - started, 3), "graph_id": id(app), "options": options}

def show_plan_run(run: dict) -> None:
    print(f"[{run['label']}] plan_hash {run['plan_hash']}, 전체 {run['seconds']}초, 겹침 {overlap_seconds(run['desk'].trace.spans)}초")
    print(timeline(run["desk"].trace.spans))
    results = {key: WorkerResult.model_validate(value) for key, value in run["results"].items()}
    show_table(join_results(results, list(results))["rows"])

print("plan_hash:", plan_hash(PLAN), "/ 작업:", [t["task_id"] for t in PLAN["tasks"]])

In [ ]:
announce("실제 Worker 3개를 Send로 병렬 실행", model_calls=12)
parallel_run = run_plan(PLAN, "병렬")
show_plan_run(parallel_run)

In [ ]:
announce("같은 PLAN, 같은 그래프를 max_concurrency=1로 실행", model_calls=12)
serial_run = run_plan(PLAN, "직렬", max_concurrency=1)
show_plan_run(serial_run)

In [ ]:
from mas_lab.evaluation import evaluate_row, load_gold, routing_table

gold = load_gold()

def plan_row(run: dict) -> dict:
    totals, spans = run["meter"].totals(), run["desk"].trace.spans
    answer = " ".join(fact for result in run["results"].values() for fact in result["facts"])
    check = evaluate_row({"structure": run["label"], "case_id": T1001["id"], "answer": answer}, gold)
    required = len(gold[T1001["id"]]["required_any"])
    return {"실행": run["label"], "config 차이": run["options"] or "없음", "plan_hash": run["plan_hash"], "전체(초)": run["seconds"], "겹침(초)": overlap_seconds(spans),
            "합산(초)": round(sum(s["end"] - s["start"] for s in spans), 2), "모델 호출": totals["model_calls"], "입력 토큰": totals["input_tokens"],
            "출력 토큰": totals["output_tokens"], "usage 누락": totals["usage_missing"], "필수 사실": f"{required - len(check['누락 사실'])}/{required}", "누락 사실": check["누락 사실"]}

print("1회 관찰입니다. 통계가 아닙니다.")
print("같은 그래프 객체:", parallel_run["graph_id"] == serial_run["graph_id"], "/ 같은 plan_hash:", parallel_run["plan_hash"] == serial_run["plan_hash"])
show_table([plan_row(parallel_run), plan_row(serial_run)])

**관찰할 것.** 이 표는 1회 관찰입니다. 위 가설을 칸별로 대조하세요.

- `plan_hash`가 두 행에서 같은지 먼저 확인합니다. 같아야 이어지는 비교에 의미가 있습니다.
- `전체(초)`와 `겹침(초)`를 함께 읽습니다. 직렬의 겹침은 0이어야 합니다. 그렇지 않다면 측정을 의심하세요.
- `합산(초)`은 각 Worker 구간의 길이를 더한 값입니다. 병렬 실행의 합산이 전체보다 훨씬 크다면, 그 차이만큼 동시에 기다린 것입니다.
- `모델 호출`과 입력 토큰은 병렬이라고 줄어드나요? 두 행이 다르다면 원인이 동시성이 아닐 수 있습니다. Worker가 도구를 몇 번 쓸지는 모델이 실행마다 정하기 때문입니다. 어느 쪽 토큰이 더 적게 나와도 동시성이 토큰을 줄였다고 읽지 마세요. `max_concurrency`는 프롬프트를 바꾸지 않습니다. 호출 하나가 더 있으면 그 호출의 입력이 통째로 더해지므로, 호출 수가 다른 두 행은 토큰으로 구조를 비교할 수 없습니다. 시간 차이도 마찬가지로 **구조만의 효과로 단정하지 마세요.** 앞의 잠자는 함수 비교는 작업이 항상 같았기 때문에 더 강하게 말할 수 있었습니다.
- 위 Worker 표의 `상태`와 `근거 수`도 두 실행에서 비교하세요. 같은 입력인데 값이 다르다면 이것이 모델 실행의 변동입니다. `complete`인데 `근거 수`가 0인 행이 있다면 스키마는 통과했지만 근거 없이 완료를 주장한 것입니다. 이 그래프는 그런 결과도 그대로 합칩니다. 막는 방법은 다음 노트북에서 다룹니다.
- `필수 사실`과 `누락 사실`은 Worker들이 낸 사실만 자동 검사한 칸입니다. 크레딧 금액을 계산하는 Worker가 이 계획에 없으므로, 금액 하나가 빠지는 것은 계획이 그렇게 생겼기 때문입니다. 다른 값까지 빠졌다면 표현 차이인지 정말 누락인지 위 Worker 표와 대조하세요.

앞 노트북에서 Supervisor가 한 메시지에 `task`를 여러 개 낼지는 **모델**이 정했습니다. 여기서는 **코드**가 정했습니다. 병렬을 지침으로 부탁하는 것과 그래프로 고정하는 것은 다른 일입니다.

그렇다고 `Send`가 곧 Multi-Agent 아키텍처는 아닙니다. `Send`는 작업을 동적으로 나누는 수단이고, 받는 쪽은 잠자는 함수일 수도 Worker일 수도 있습니다. 앞에서 함수로 먼저 돌려 본 이유입니다.

다음 절은 "어느 쪽으로 갈지"를 묻습니다. T-1001 같은 복합 요청만 있다면 언제나 세 Worker를 돌리면 되지만, "신청 기한이 언제인가요?"에도 세 Worker를 돌릴 필요는 없습니다.

## 직접 해보기

수강생이 자기 업무로 바꿔 보는 칸입니다. 해보기 3만 모델을 부르고 `RUN_EXERCISE = False`로 꺼져 있습니다. 나머지는 고친 뒤 그대로 실행하면 됩니다.

### 해보기 1

자기 업무를 `TASKS` 형식으로 쓰고 동시에 실행 가능한 묶음을 찾으세요. 아래 예시는 월간 보고서 작성입니다. `needs`를 바꾸면서 묶음이 어떻게 달라지는지 보세요. 일부러 순환을 넣어 거절 메시지도 확인해 보세요.

In [ ]:
MY_TASKS = {
    "수집": {"needs": [], "state": "raw_data", "done": "이번 달 자료가 모였다"},
    "매출 분석": {"needs": ["수집"], "state": "sales", "done": "전월과 비교했다"},
    "장애 분석": {"needs": ["수집"], "state": "incidents", "done": "주요 장애를 정리했다"},
    "초안": {"needs": ["매출 분석", "장애 분석"], "state": "draft", "done": "요약 표가 들어갔다"},
    "승인": {"needs": ["초안"], "state": "approved", "done": "담당자가 승인했다"},
}
for number, batch in enumerate(plan_batches(MY_TASKS), 1):
    print(f"묶음 {number}: {batch}")
print("모델 호출 0회")

### 해보기 2

병렬 이득이 사라지는 작업 시간 분포를 찾으세요. `MY_WORK`의 값을 바꾸며 `이득` 칸을 읽고, 이득이 작아지는 이유를 최장 경로로 설명하세요. 한 작업이 합계의 대부분을 차지할 때 어떻게 되나요?

In [ ]:
MY_WORK = {"billing": 0.05, "policy": 0.05, "outage": 0.6}
parallel, serial = run_fan(MY_WORK), run_fan(MY_WORK, max_concurrency=1)
show_table([{"plan_hash": plan_hash(MY_WORK), "최장 경로(초)": longest_path(MY_WORK, {}), "병렬 실측(초)": parallel["seconds"], "직렬 실측(초)": serial["seconds"],
             "이득(초)": round(serial["seconds"] - parallel["seconds"], 3), "이득 비율": round(1 - parallel["seconds"] / serial["seconds"], 2)}])
print("모델 호출 0회")

### 해보기 3

작업을 합쳐도 필수 사실이 유지되는지 봅니다. `MERGED`는 `billing`과 `outage`를 한 Worker에게 맡깁니다. 묶는 방법을 바꿔 보고, 앞의 비교표와 `필수 사실`, 입력 토큰을 비교하세요. 같은 Worker가 도구를 함께 받으면 호출 수가 줄어드나요, Context가 커지나요? 이 셀은 실제로 모델을 부릅니다.

In [ ]:
RUN_EXERCISE = False
MERGED = {"facts": ["billing", "outage"], "policy": ["policy"]}

def merged_worker(payload: dict, config: RunnableConfig):
    desk, task = config["configurable"]["desk"], WorkerTask.model_validate(payload["task"])
    allowed = frozenset(task.allowed_tools)
    agent = create_deep_agent(model=model, tools=[t for t in desk.tools("all") if t.name in allowed], system_prompt="주어진 작업 계약을 따르는 조사 Worker입니다.",
                              middleware=allowlist_middleware(desk.budget, allowed), permissions=DENY_WRITES, response_format=ProviderStrategy(WorkerResult), name=f"worker-{task.task_id}")
    with desk.trace.span(task.task_id, "worker"):
        state = agent.invoke({"messages": [{"role": "user", "content": contract_prompt(task, desk.scope)}]}, config={"recursion_limit": 40})
    return {"results": {task.task_id: state["structured_response"].model_copy(update={"task_id": task.task_id}).model_dump()}}

if not RUN_EXERCISE:
    print("실행하지 않았습니다. 고친 뒤 RUN_EXERCISE를 True로 바꾸세요.")
else:
    announce("작업을 합친 계획으로 다시 실행", model_calls=8)
    goal = lambda roles: "\n".join(ROLE_GOALS[role] for role in roles) + f"\n고객 요청: {T1001['question']}"
    merged_plan = {**PLAN, "tasks": [{"task_id": name, "goal": goal(roles), "allowed_tools": sorted(frozenset().union(*(ROLE_TOOLS[r] for r in roles)))} for name, roles in MERGED.items()]}
    merged_run = run_plan(merged_plan, "합친 계획", app=build_plan_app(merged_worker))
    show_plan_run(merged_run)
    show_table([plan_row(parallel_run), plan_row(merged_run)])

## 정리

| 흐름 | 그래프 도구 | 막는 실패 | 재현한 실패 |
|---|---|---|---|
| 순차 | `add_edge`, 전달물 | 선행 결과 없는 실행 | 범위 밖 고객에게 `credit` 0원 |
| 병렬 | `Send`, reducer, `max_concurrency` | 쓸데없는 대기 | `InvalidUpdateError`, 의존 무시로 빈 입력 |

병렬의 시간 효과는 같은 분해 안에서만 비교합니다. `plan_hash`가 같은 계획을 `max_concurrency`만 바꿔 돌린 것이 이 노트북의 비교 방법입니다. 줄어드는 것은 전체 시간이고, 호출과 토큰이 줄어든다고 기대할 근거는 없습니다.

순차와 병렬은 **업무의 선행 조건만으로** 정해집니다. 그런데 어떤 작업을 돌릴지가 요청의 내용에 달려 있거나, 결과가 모자라면 다시 해야 한다면 선행 조건 표만으로는 그래프를 그릴 수 없습니다. 그 두 경우가 03b의 분기와 루프입니다.

## 확인 질문

1. `Send`로 fan-out했는데 reducer가 필요한 이유는 무엇인가요? 3절의 실패 재현에서 본 `InvalidUpdateError` 문장으로 답하세요.
2. 일반 함수로 돌린 `max_concurrency=1` 비교가 실제 Worker 비교표보다 병렬 효과에 대해 더 강하게 말할 수 있는 이유는 무엇인가요?
3. 병렬인데 토큰이 줄지 않는 이유는 무엇이고, 그래도 병렬을 택할 조건은 무엇인가요?

다음 노트북 03b는 분기와 루프를 만듭니다. 요청마다 다른 경로로 보내고, 모자라면 되돌아가는 그래프에서 무엇이 틀릴 수 있는지를 봅니다.